In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
NUM_FRAMES = 15
BATCH_SIZE = 4
EPOCHS = 10

In [2]:
def load_video_frames(video_path):
    frames = sorted(os.listdir(video_path))[:NUM_FRAMES]
    video = []

    for frame in frames:
        img_path = os.path.join(video_path, frame)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        video.append(img)

    # Padding if less frames
    while len(video) < NUM_FRAMES:
        video.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

    return np.array(video, dtype=np.float32)


def generator(root_dir):
    classes = ["Celeb-real", "Celeb-synthesis"]

    for label, cls in enumerate(classes):
        class_path = os.path.join(root_dir, cls)

        for video_folder in os.listdir(class_path):
            video_path = os.path.join(class_path, video_folder)

            if os.path.isdir(video_path):
                video_tensor = load_video_frames(video_path)
                yield video_tensor, label

In [3]:
output_signature = (
    tf.TensorSpec(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\praveen tamminaina\\Downloads\\30 DEC TZ\\Deepfake\\Deepfake-detection\\dataset_faces\\dataset_faces\\train"),
    output_signature=output_signature
).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\praveen tamminaina\\Downloads\\30 DEC TZ\\Deepfake\\Deepfake-detection\\dataset_faces\\dataset_faces\\val"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\praveen tamminaina\\Downloads\\30 DEC TZ\\Deepfake\\Deepfake-detection\\dataset_faces\\dataset_faces\\test"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [4]:
class PositionalEmbedding(layers.Layer):

    def __init__(self, num_frames, embed_dim):
        super().__init__()
        self.pos_emb = layers.Embedding(
            input_dim=num_frames,
            output_dim=embed_dim
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[1], delta=1)
        position_embeddings = self.pos_emb(positions)
        return x + position_embeddings

In [5]:
def build_model():

    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling="avg"
    )
    backbone.trainable = False

    video_input = layers.Input(
        shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3)
    )

    x = layers.TimeDistributed(backbone)(video_input)

    #  ADD POSITIONAL ENCODING HERE
    x = PositionalEmbedding(NUM_FRAMES, 1280)(x)

    # Transformer block
    x1 = layers.LayerNormalization()(x)

    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=128
    )(x1, x1)

    x2 = layers.Add()([x, attention])   # OK same shape

    x3 = layers.LayerNormalization()(x2)

    ff = layers.Dense(512, activation="relu")(x3)
    ff = layers.Dense(1280)(ff)   # 🔥 IMPORTANT: back to 1280

    x4 = layers.Add()([x2, ff])   # Now shapes match

    x = layers.GlobalAveragePooling1D()(x4)

    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(video_input, output)

    return model

In [6]:
model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 15, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 15, 1280)  │  4,049,571 │ input_layer_1[0]… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 15, 1280)  │     19,200 │ time_distributed… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 15, 1280)  │      2,560 │ positional_embed… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 15, 1280)  │  2,624,256 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 15, 1280)  │          0 │ positional_embed… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 15, 1280)  │      2,560 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 15, 512)   │    655,872 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 15, 1280)  │    656,640 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 15, 1280)  │          0 │ add[0][0],        │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ add_1[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │      1,281 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,011,940 (30.56 MB)

 Trainable params: 3,962,369 (15.12 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [7]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2),
    tf.keras.callbacks.ModelCheckpoint("best_vit_model.h5", save_best_only=True)
]

In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

TRAIN_SAMPLES = 4570
VAL_SAMPLES = 978

model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    steps_per_epoch = TRAIN_SAMPLES // BATCH_SIZE,
    validation_steps = VAL_SAMPLES // BATCH_SIZE,
    callbacks = callbacks
)


Epoch 1/10


  92/1142 ━━━━━━━━━━━━━━━━━━━━ 31:23 2s/step - accuracy: 1.0000 - loss: 0.0359

KeyboardInterrupt: 

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
def make_gradcam_heatmap(model, video_tensor, layer_name="top_conv"):

    # Take one frame only for GradCAM
    frame = video_tensor[0:1, 0]  # first frame

    backbone = model.layers[1].layer  # EfficientNet

    grad_model = keras.Model(
        [backbone.inputs],
        [backbone.get_layer(layer_name).output, backbone.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(frame)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
import matplotlib.pyplot as plt

sample_video, label = next(iter(test_dataset))
heatmap = make_gradcam_heatmap(model, sample_video)

plt.matshow(heatmap)
plt.colorbar()
plt.show()